In [1]:
import pandas as pd

# Đọc dữ liệu từ file parquet
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")

# Chuyển đổi timestamp sang số (giây)
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

# Sắp xếp tĩnh trực tiếp theo thời gian
train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

print("Dữ liệu đã được load và sắp xếp theo thời gian thành công!")


Dữ liệu đã được load và sắp xếp theo thời gian thành công!


In [2]:
import numpy as np
import pandas as pd
import torch

def create_graph_snapshots(df, feature_cols, label_col='label', window_size=2000, overlap=1500):
    """
    Cắt DataFrame thành các Graph Snapshots dựa trên cửa sổ trượt (mỗi snapshot chứa window_size luồng, và có overlap luồng với snapshot trước đó).
    
    Args:
        df (pd.DataFrame): DataFrame đã được sort theo timestamp.
        feature_cols (list): Danh sách các cột đặc trưng hành vi làm node features.
        label_col (str): Tên cột chứa nhãn tấn công.
        window_size (int): Số lượng luồng (flow) trong một snapshot.
        overlap (int): Số lượng luồng giữ lại từ snapshot trước đó.
        
    Returns:
        list: Danh sách các dictionary, mỗi dict là một snapshot chứa 'x', 'y', và 'meta'.
    """
    snapshots = []
    
    # Tính bước trượt (stride)
    stride = window_size - overlap
    if stride <= 0:
        raise ValueError("Overlap phải nhỏ hơn window_size!")
        
    num_flows = len(df)
    

    X_all = df[feature_cols].values.astype(np.float32)
    Y_all = df[label_col].values.astype(np.int64)
    
    # Metadata cho việc nối cạnh
    dst_ips = df['dst_ip'].values
    dst_ports = df['dst_port'].values
    timestamps = df['timestamp_num'].values
    
    print(f"Bắt đầu tạo snapshot: Tổng số {num_flows} flows, Window={window_size}, Stride={stride}")
    
    # Quét qua toàn bộ dữ liệu với bước nhảy là stride
    for start_idx in range(0, num_flows - window_size + 1, stride):
        end_idx = start_idx + window_size
        
        # 1. Trích xuất Node Features và Labels cho Snapshot hiện tại
        x_snapshot = X_all[start_idx:end_idx]
        y_snapshot = Y_all[start_idx:end_idx]
        
        # 2. Trích xuất Metadata tương ứng
        meta_snapshot = {
            'dst_ip': dst_ips[start_idx:end_idx],
            'dst_port': dst_ports[start_idx:end_idx],
            'timestamp': timestamps[start_idx:end_idx],
            'global_indices': np.arange(start_idx, end_idx) # Lưu lại index gốc để track
        }
        
        # 3. Đóng gói vào dictionary
        snapshot_data = {
            'x': torch.tensor(x_snapshot),
            'y': torch.tensor(y_snapshot),
            'meta': meta_snapshot
        }
        
        snapshots.append(snapshot_data)
        
    print(f"Đã tạo thành công {len(snapshots)} snapshots!")
    return snapshots

In [3]:
# tạo tham số cho cửa sổ trượt
WINDOW_SIZE = 2000
OVERLAP = 1500
feature_cols = [col for col in train_df.columns if col not in ["flow_id",'timestamp', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'label', 'timestamp_num', "src_port", "dst_port"]]
print(f"Các cột đặc trưng được sử dụng: {len(feature_cols)} cột")
# Tạo snapshots cho tập Train
train_snapshots = create_graph_snapshots(
    df=train_df, 
    feature_cols=feature_cols, 
    label_col='label', 
    window_size=WINDOW_SIZE, 
    overlap=OVERLAP
)

# Tạo snapshots cho tập Valid và Test
valid_snapshots = create_graph_snapshots(
    df=valid_df, 
    feature_cols=feature_cols, 
    label_col='label', 
    window_size=WINDOW_SIZE, 
    overlap=OVERLAP
)

test_snapshots = create_graph_snapshots(
    df=test_df, 
    feature_cols=feature_cols, 
    label_col='label', 
    window_size=WINDOW_SIZE, 
    overlap=OVERLAP
)

Các cột đặc trưng được sử dụng: 314 cột
Bắt đầu tạo snapshot: Tổng số 2470638 flows, Window=2000, Stride=500
Đã tạo thành công 4938 snapshots!
Bắt đầu tạo snapshot: Tổng số 570134 flows, Window=2000, Stride=500
Đã tạo thành công 1137 snapshots!
Bắt đầu tạo snapshot: Tổng số 760240 flows, Window=2000, Stride=500
Đã tạo thành công 1517 snapshots!


In [4]:
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from tqdm import tqdm  # Thư viện tạo thanh tiến trình chuyên nghiệp

def build_graph_from_snapshot(snapshot, k=10, alpha=0.7, beta=0.3, lambda_decay=3.0):
    """
    Nối cạnh cho một snapshot đơn lẻ
    Args:
        snapshot (dict): Dictionary chứa "x", "y và "meta"
        k (int): số lượng láng giềng tối đa có thể nối
        beta: hệ số trọng số cho Time Decay
        lambda_decay: hệ số điều chỉnh độ nhạy của Time Decay
    """
    x = snapshot['x']
    y = snapshot['y']
    meta = snapshot['meta']
    num_nodes = x.shape[0]

    target_ids = np.array([f"{ip}:{port}" for ip, port in zip(meta['dst_ip'], meta['dst_port'])])
    timestamps = meta['timestamp']
    edge_index_list = []

    # vòng lặp nối cạnh: nối cạnh nếu cùng dst_ip và dst_port, ưu tiên những node có timestamp gần nhau nhất
    # weight cạnh = alpha * cosine_similarity + beta * time_decay
    for i in range(num_nodes):
        same_target_mask = (target_ids == target_ids[i])
        same_target_indices = np.where(same_target_mask)[0]
        same_target_indices = same_target_indices[same_target_indices != i]

        if len(same_target_indices) == 0:
            continue

        time_diffs = np.abs(timestamps[same_target_indices] - timestamps[i])
        k_actual = min(k, len(time_diffs))
        top_k_local_indices = np.argsort(time_diffs)[:k_actual]
        top_k_global_indices = same_target_indices[top_k_local_indices]

        for j in top_k_global_indices:
            edge_index_list.append([i, j])

    if not edge_index_list:
        return Data(x=x, edge_index=torch.empty((2, 0), dtype=torch.long), 
                    edge_attr=torch.empty((0,), dtype=torch.float32), y=y)

    edge_index = torch.tensor(edge_index_list, dtype=torch.long).t().contiguous()

    # Tính Cosine Similarity
    src_nodes_features = x[edge_index[0]]
    dst_nodes_features = x[edge_index[1]]
    cos_sim = F.cosine_similarity(src_nodes_features, dst_nodes_features, dim=1, eps=1e-8)
    cos_sim = (cos_sim + 1.0) / 2.0 

    # Tính Time Decay với chuẩn hóa Min-Max cục bộ (để phù hợp với range của cosine similarity)
    src_times = torch.tensor(timestamps[edge_index[0].numpy()], dtype=torch.float32)
    dst_times = torch.tensor(timestamps[edge_index[1].numpy()], dtype=torch.float32)
    time_diffs_tensor = torch.abs(src_times - dst_times)
    
    max_diff = time_diffs_tensor.max()
    if max_diff > 0:
        time_diffs_norm = time_diffs_tensor / max_diff
    else:
        time_diffs_norm = time_diffs_tensor
        
    time_decay = torch.exp(-lambda_decay * time_diffs_norm)
    edge_weight = alpha * cos_sim + beta * time_decay

    return Data(x=x, edge_index=edge_index, edge_attr=edge_weight, y=y)


def convert_all_snapshots_to_graphs(snapshots_list, dataset_name="Train Set", k=10, alpha=0.7, beta=0.3, lambda_decay=3.0):
    """
    Hàm wrapper bọc tqdm bên ngoài để track tiến trình tạo đồ thị cho danh sách snapshots.
    Args:
        snapshots_list (list): Danh sách các snapshot (mỗi snapshot là một dict chứa "x", "y", "meta")
        dataset_name (str): Tên của tập dữ liệu (dùng để hiển thị trên thanh tiến trình)
        k, alpha, beta, lambda_decay: Tham số cho hàm build_graph_from_snapshot
    """
    graph_objects = []
    
    # tqdm sẽ tự động tính toán % tiến độ, số snapshot/s và thời gian còn lại
    progress_bar = tqdm(
        snapshots_list, 
        desc=f"Building Graphs ({dataset_name})", 
        unit="snapshot",
        ncols=100 # Độ rộng của thanh tiến trình hiển thị trên console
    )
    
    for snap in progress_bar:
        graph_data = build_graph_from_snapshot(
            snapshot=snap, 
            k=k, 
            alpha=alpha, 
            beta=beta, 
            lambda_decay=lambda_decay
        )
        graph_objects.append(graph_data)
        
    return graph_objects

In [5]:
# tạo đồ thị cho các tập train, valid và test
train_graphs = convert_all_snapshots_to_graphs(train_snapshots, dataset_name="Train Set")


valid_graphs = convert_all_snapshots_to_graphs(valid_snapshots, dataset_name="Valid Set")


test_graphs = convert_all_snapshots_to_graphs(test_snapshots, dataset_name="Test Set")

Building Graphs (Test Set): 100%|█████████████████████████| 1517/1517 [05:50<00:00,  4.32snapshot/s]


In [6]:
from torch_geometric.loader import DataLoader

# Thiết lập Batch Size (Số lượng snapshot đưa vào GPU trong 1 bước)
BATCH_SIZE = 32 

# Khởi tạo DataLoader cho cả 3 tập
# Tập Train cần shuffle=True để mô hình học khách quan, Valid/Test giữ nguyên thứ tự
train_loader = DataLoader(train_graphs, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_graphs, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_graphs, batch_size=BATCH_SIZE, shuffle=False)

print(f"Đã chuẩn bị xong DataLoader:")
print(f" - Train Batches: {len(train_loader)}")
print(f" - Valid Batches: {len(valid_loader)}")
print(f" - Test Batches: {len(test_loader)}")

Đã chuẩn bị xong DataLoader:
 - Train Batches: 155
 - Valid Batches: 36
 - Test Batches: 48


In [7]:
import torch
import numpy as np

# class EarlyStopping để theo dõi macro f1 trên tập validation
class EarlyStopping:
    """
    Class EarlyStopping theo macro f1 trên tập validation
    """
    def __init__(self, patience=8, path='best_baseline_gcn.pt'):
        self.patience = patience
        self.path = path
        self.counter = 0
        self.best_f1 = -float('inf')
        self.early_stop = False

    # hàm call: mỗi khi object(val_f1, model) được gọi thì sẽ chạy hàm này
    def __call__(self, val_f1, model):
        # Nếu F1 cải thiện (lớn hơn điểm tốt nhất trước đó)
        if val_f1 > self.best_f1:
            self.best_f1 = val_f1
            self.save_checkpoint(model)
            self.counter = 0 # Reset bộ đếm về 0
        else:
            # Nếu không cải thiện, tăng bộ đếm thêm 1
            self.counter += 1
            print(f' -> [EarlyStopping] Bộ đếm tăng: {self.counter} / {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True

    def save_checkpoint(self, model):
        '''Lưu mô hình khi điểm Validation F1 tăng'''
        torch.save(model.state_dict(), self.path)
        print(f' -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.')

In [8]:
import torch
import numpy as np
from tqdm import tqdm
from sklearn.metrics import f1_score


def train_epoch(epoch, EPOCHS):
    """Hàm huấn luyện cho một epoch

    Args:
        epoch (int): số epoch hiện tại
        EPOCHS (int): số epoch tổng cộng để huấn luyện (dùng để hiển thị trên thanh tiến trình)

    Returns:
        loss_avg (float): loss trung bình trên tập train
        macro_f1 (float): điểm Macro F1 trên tập train
    """
    
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    # Thanh tiến trình cho tập Train
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch:03d}/{EPOCHS} [Train]", leave=False, ncols=100)
    
    for batch in progress_bar:
        batch = batch.to(device)
        optimizer.zero_grad()
        
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        loss = criterion(out, batch.y)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * batch.num_graphs
        pred = out.argmax(dim=1)
        
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())
        
        # Cập nhật số loss liên tục trên thanh tiến trình
        progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    loss_avg = total_loss / len(train_loader.dataset)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return loss_avg, macro_f1

@torch.no_grad()
def valid_epoch(epoch, EPOCHS):
    """Hàm đánh giá trên tập validation

    Args:
        epoch (int): số epoch hiện tại
        EPOCHS (int): số epoch tổng cộng để huấn luyện (dùng để hiển thị trên thanh tiến trình)

    Returns:
        loss_avg (float): loss trung bình trên tập validation
        macro_f1 (float): điểm Macro F1 trên tập validation
    """
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    # Thanh tiến trình cho tập Valid
    progress_bar = tqdm(valid_loader, desc=f"Epoch {epoch:03d}/{EPOCHS} [Valid]", leave=False, ncols=100)
    
    for batch in progress_bar:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        loss = criterion(out, batch.y)
        
        total_loss += loss.item() * batch.num_graphs
        pred = out.argmax(dim=1)
        
        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(batch.y.cpu().numpy())
        
    loss_avg = total_loss / len(valid_loader.dataset)
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return loss_avg, macro_f1

In [9]:
from sklearn.metrics import f1_score, roc_auc_score
from tqdm import tqdm
import torch

@torch.no_grad()
def test_benchmark_new_nodes(loader, window_size=2000, stride=500):
    """Hàm đánh giá trên tập Test với chiến lược chỉ đánh giá luồng mới (bỏ qua phần overlap)
    Args:
        loader: DataLoader cho tập Test
        window_size: kích thước cửa sổ (số node trong mỗi snapshot)
        stride: bước nhảy giữa các snapshot (window_size - overlap)
    Returns:
        final_macro_f1 (float): điểm Macro F1 cuối cùng trên tập Test
        auc_roc (float): điểm AUC-ROC cuối cùng trên tập Test
        all_preds (list): danh sách dự đoán của mô hình cho tất cả node được đánh giá
        all_labels (list): danh sách nhãn thực tế tương ứng với các node
        all_probs (list): danh sách xác suất dự đoán
    """
    model.eval() # Tắt tính toán gradient, tắt Dropout
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    overlap = window_size - stride
    curr_snap_idx = 0
    
    print("Đang chạy Benchmark trên tập Test (Chiến lược: Chỉ đánh giá luồng Mới)...")
    progress_bar = tqdm(loader, desc="Test Benchmark", ncols=100)
    
    for batch in progress_bar:
        batch = batch.to(device)
        
        # Chạy forward pass qua mô hình
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        
        # Dự đoán class có xác suất cao nhất ngay lập tức
        preds = out.argmax(dim=1)
        probs = torch.exp(out) # Hàm loss là NLLLoss kết hợp với log_softmax, nên lấy exp để ra xác suất
        
        # Bóc tách từng đồ thị trong batch để cắt lấy các Node mới
        for g_idx in range(batch.num_graphs):
            # Tạo mask để lấy đúng các node thuộc về đồ thị thứ g_idx
            node_mask = (batch.batch == g_idx)
            
            g_preds = preds[node_mask]
            g_probs = probs[node_mask]
            g_labels = batch.y[node_mask]
            g_num_nodes = g_preds.size(0)
            
            # Logic cắt node mới
            if curr_snap_idx == 0:
                # Snapshot đầu tiên: Đánh giá toàn bộ
                start_eval_idx = 0
            else:
                # Các Snapshot sau: Bỏ qua phần overlap, chỉ lấy phần mới
                start_eval_idx = overlap
                
            # Đề phòng trường hợp snapshot cuối cùng bị thiếu node
            if start_eval_idx < g_num_nodes:
                new_preds = g_preds[start_eval_idx:g_num_nodes]
                new_probs = g_probs[start_eval_idx:g_num_nodes]
                new_labels = g_labels[start_eval_idx:g_num_nodes]
                
                # Nạp vào danh sách tổng
                all_preds.extend(new_preds.cpu().numpy())
                all_probs.extend(new_probs.cpu().numpy())
                all_labels.extend(new_labels.cpu().numpy())
            
            curr_snap_idx += 1
            
    # Tính F1 Score cuối cùng
    final_macro_f1 = f1_score(all_labels, all_preds, average='macro')
    auc_roc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')
    
    print(f"Tổng số flow thực tế được đánh giá: {len(all_labels)}")
    print(f"=====================================")
    print(f"🏆 CHÍNH THỨC - TEST BENCHMARK MACRO F1: {final_macro_f1:.4f}")
    print(f"🏆 CHÍNH THỨC - TEST BENCHMARK AUC-ROC: {auc_roc:.4f}")
    print(f"=====================================")

    # in ra classification report chi tiết
    from sklearn.metrics import classification_report
    print("\n📊 Classification Report Chi Tiết:")
    print(classification_report(all_labels, all_preds, digits=4))
    
    return final_macro_f1, auc_roc, all_preds, all_labels, all_probs

Thử bằng model GAT

In [10]:
import torch
import torch.nn.functional as F
from torch.nn import Linear
from torch_geometric.nn import GATConv

# Thay các lớp GCNConv bằng GATConv, đồng thời thêm tham số edge_attr để tận dụng trọng số cạnh đã tính toán
class AdvancedGAT(torch.nn.Module):
    def __init__(self, num_node_features, num_classes=11):
        super(AdvancedGAT, self).__init__()
        
        # 1. Lớp đầu vào (Giữ nguyên để nén feature)
        self.lin_in = Linear(num_node_features, 32)
        
        # 2. Các lớp GAT với Multi-head Attention
        # Lớp 1: Input 32 -> 4 heads x 16 = 64 chiều (Tương đương GCNConv 64)
        self.gat1 = GATConv(
            in_channels=32, out_channels=16, heads=4, concat=True, edge_dim=1
        )
        
        # Lớp 2: Input 64 -> 4 heads x 32 = 128 chiều (Tương đương GCNConv 128)
        self.gat2 = GATConv(
            in_channels=64, out_channels=32, heads=4, concat=True, edge_dim=1
        )
        
        # Lớp 3: Input 128 -> 4 heads x 64. 
        # CHÚ Ý: Ở lớp GNN cuối cùng, ta dùng concat=False để tính trung bình các heads, 
        # gom lại thành 64 chiều chuẩn bị cho lớp Linear.
        self.gat3 = GATConv(
            in_channels=128, out_channels=64, heads=4, concat=False, edge_dim=1
        )
        
        # 3. Lớp phân loại đầu ra
        self.lin_out = Linear(64, num_classes)

    def forward(self, x, edge_index, edge_attr):
        # GATConv yêu cầu edge_attr phải là ma trận 2 chiều [Số cạnh, Số feature cạnh].
        # Trọng số Cosine + Time Decay của ta đang là mảng 1 chiều [E], ta cần thêm 1 trục (unsqueeze).
        if edge_attr.dim() == 1:
            edge_attr = edge_attr.unsqueeze(-1)
            
        x = self.lin_in(x)
        x = F.relu(x)
        
        # Lớp GAT 1 (activation là elu thay vì relu để phù hợp với paper GAT gốc)
        x = self.gat1(x, edge_index, edge_attr=edge_attr)
        x = F.elu(x) 
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Lớp GAT 2
        x = self.gat2(x, edge_index, edge_attr=edge_attr)
        x = F.elu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Lớp GAT 3
        x = self.gat3(x, edge_index, edge_attr=edge_attr)
        x = F.elu(x)
        x = F.dropout(x, p=0.5, training=self.training)
        
        # Lớp Output
        x = self.lin_out(x)
        
        return F.log_softmax(x, dim=1)

In [12]:
# 1. Khởi tạo GAT Model
model = AdvancedGAT(num_node_features=314, num_classes=11)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# 2. Khởi tạo lại Optimizer (Bắt buộc phải tạo lại để nó theo dõi trọng số của GAT)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)



# 3. Reset Early Stopping (Nên đổi tên file lưu để không ghi đè mất file GCN cũ)
early_stopping = EarlyStopping(patience=8, path='best_advanced_gat.pt')

print(model)

AdvancedGAT(
  (lin_in): Linear(in_features=314, out_features=32, bias=True)
  (gat1): GATConv(32, 16, heads=4)
  (gat2): GATConv(64, 32, heads=4)
  (gat3): GATConv(128, 64, heads=4)
  (lin_out): Linear(in_features=64, out_features=11, bias=True)
)


In [14]:
# Khởi tạo mô hình

# Chuyển mô hình lên GPU (nếu có)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Khai báo Optimizer (Adam là chuẩn mực nhất)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

# 1. Chạy đoạn code sinh class_weights của bạn
# LƯU Ý: all_train_labels phải là nhãn của TẬP TRAIN (Không gộp Valid/Test vào để tránh Data Leakage)
import sklearn.utils.class_weight as class_weight

unique_classes = np.unique(train_df['label']) # Lấy các class có trong tập train
all_train_labels = train_df['label'].values

class_weights = class_weight.compute_class_weight(
    class_weight='balanced', 
    classes=unique_classes, 
    y=all_train_labels
)
class_weights = np.nan_to_num(class_weights, nan=1.0, posinf=10.0, neginf=1.0)
class_weights = np.power(class_weights, 0.4) 
class_weights = np.clip(class_weights, a_min=0.5, a_max=6.0) 

# Đưa lên thiết bị (GPU/CPU)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device) 

# ==========================================
# 2. Áp dụng vào Hàm Mất Mát (Loss Function)
# ==========================================
# Chèn class_weights_tensor vào tham số 'weight' của NLLLoss
criterion = torch.nn.NLLLoss(weight=class_weights_tensor)

print("Đã tích hợp Class Weights vào hàm Loss thành công!")
print(f"Trọng số các class: {class_weights}")


Đã tích hợp Class Weights vào hàm Loss thành công!
Trọng số các class: [1.64730877 0.5        3.7653502  1.29446194 1.67705869 3.80442181
 2.0250111  0.83822433 1.76304403 1.22605595 1.69183888]


In [15]:
EPOCHS = 50


early_stopping = EarlyStopping(patience=8, path='best_baseline_gcn.pt')

print("Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...")

for epoch in range(1, EPOCHS + 1):
    # 1. Chạy huấn luyện và lấy kết quả
    train_loss, train_f1 = train_epoch(epoch, EPOCHS)
    
    # 2. Chạy validation đánh giá
    valid_loss, valid_f1 = valid_epoch(epoch, EPOCHS)
    
    # In kết quả của epoch hiện tại
    print(f"Epoch: {epoch:03d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} - Macro F1: {train_f1:.4f} | "
          f"Valid Loss: {valid_loss:.4f} - Macro F1: {valid_f1:.4f}")
    
    # 3. Kiểm tra điều kiện dừng sớm
    early_stopping(valid_f1, model)
      
    if early_stopping.early_stop:
        print(f"\n🛑 [DỪNG SỚM] Mô hình không cải thiện sau {early_stopping.patience} Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch {epoch}!")
        break

Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...


Epoch: 001/50 | Train Loss: 0.8905 - Macro F1: 0.4174 | Valid Loss: 0.3747 - Macro F1: 0.6289
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 002/50 | Train Loss: 0.3533 - Macro F1: 0.7483 | Valid Loss: 0.1919 - Macro F1: 0.8427
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 003/50 | Train Loss: 0.1621 - Macro F1: 0.8707 | Valid Loss: 0.1371 - Macro F1: 0.8873
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 004/50 | Train Loss: 0.1876 - Macro F1: 0.8874 | Valid Loss: 0.1170 - Macro F1: 0.8978
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


...

Epoch: 031/50 | Train Loss: 0.0258 - Macro F1: 0.9807 | Valid Loss: 0.1060 - Macro F1: 0.9300
 -> [EarlyStopping] Bộ đếm tăng: 4 / 8


Epoch: 032/50 | Train Loss: 0.0211 - Macro F1: 0.9826 | Valid Loss: 0.0884 - Macro F1: 0.9319
 -> [EarlyStopping] Bộ đếm tăng: 5 / 8


Epoch: 033/50 | Train Loss: 0.0265 - Macro F1: 0.9795 | Valid Loss: 0.1200 - Macro F1: 0.9234
 -> [EarlyStopping] Bộ đếm tăng: 6 / 8


Epoch: 034/50 | Train Loss: 0.0302 - Macro F1: 0.9774 | Valid Loss: 0.1136 - Macro F1: 0.9006
 -> [EarlyStopping] Bộ đếm tăng: 7 / 8


Epoch: 035/50 | Train Loss: 0.0271 - Macro F1: 0.9702 | Valid Loss: 0.1092 - Macro F1: 0.9320
 -> [EarlyStopping] Bộ đếm tăng: 8 / 8

🛑 [DỪNG SỚM] Mô hình không cải thiện sau 8 Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch 35!


In [16]:
# 1. Nạp lại trọng số tối ưu nhất đã được Early Stopping lưu lại trước đó
model.load_state_dict(torch.load('best_baseline_gcn.pt'))
model = model.to(device)
print("Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!")

# 2. Chạy Benchmark chính thức trên tập Test (Chỉ lấy node mới, không tính trùng overlap)
test_f1, test_auc, test_preds, test_labels, test_probs = test_benchmark_new_nodes(
    loader=test_loader, 
    window_size=2000, 
    stride=500
)

C:\Users\Admin\AppData\Local\Temp\ipykernel_20084\2950129650.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_baseline_gcn.pt'))


Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!
Đang chạy Benchmark trên tập Test (Chiến lược: Chỉ đánh giá luồng Mới)...


Test Benchmark: 100%|███████████████████████████████████████████████| 48/48 [00:29<00:00,  1.60it/s]


Tổng số flow thực tế được đánh giá: 760000
🏆 CHÍNH THỨC - TEST BENCHMARK MACRO F1: 0.8542
🏆 CHÍNH THỨC - TEST BENCHMARK AUC-ROC: 0.9975

📊 Classification Report Chi Tiết:
              precision    recall  f1-score   support

           0     0.8375    0.8948    0.8652     19846
           1     0.9964    1.0000    0.9982    484077
           2     0.5678    0.9789    0.7187      2515
           3     0.9989    0.8536    0.9206     36253
           4     0.6008    0.8165    0.6923     18979
           5     0.9867    0.9996    0.9931      2451
           6     0.6209    0.7007    0.6584     11847
           7     1.0000    0.9950    0.9975    107196
           8     0.6864    0.9890    0.8104     16746
           9     1.0000    0.6668    0.8001     41523
          10     0.9688    0.9158    0.9416     18567

    accuracy                         0.9597    760000
   macro avg     0.8422    0.8919    0.8542    760000
weighted avg     0.9684    0.9597    0.9607    760000



In [17]:
# 1. Khởi tạo GAT Model

model = AdvancedGAT(num_node_features=314, num_classes=11)
model = model.to(device)

# 2. Khởi tạo lại Optimizer (Bắt buộc phải tạo lại để nó theo dõi trọng số của GAT)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

# (Hàm criterion NLLLoss với class_weights của bạn vẫn giữ nguyên không đổi)

# 3. Reset Early Stopping (Nên đổi tên file lưu để không ghi đè mất file GCN cũ)
early_stopping = EarlyStopping(patience=8, path='best_advanced_gat.pt')

print(model)
EPOCHS = 50

# Khởi tạo bộ quản lý Early Stopping với patience = 8
early_stopping = EarlyStopping(patience=8, path='best_baseline_gcn.pt')

print("Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...")
for epoch in range(1, EPOCHS + 1):
    # 1. Chạy huấn luyện và lấy kết quả
    train_loss, train_f1 = train_epoch(epoch, EPOCHS)
    
    # 2. Chạy validation đánh giá
    valid_loss, valid_f1 = valid_epoch(epoch, EPOCHS)
    
    # In kết quả của epoch hiện tại
    print(f"Epoch: {epoch:03d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} - Macro F1: {train_f1:.4f} | "
          f"Valid Loss: {valid_loss:.4f} - Macro F1: {valid_f1:.4f}")
    
    # 3. Kiểm tra điều kiện dừng sớm
    early_stopping(valid_f1, model)
      
    if early_stopping.early_stop:
        print(f"\n🛑 [DỪNG SỚM] Mô hình không cải thiện sau {early_stopping.patience} Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch {epoch}!")
        break
    
model.load_state_dict(torch.load('best_baseline_gcn.pt'))
model = model.to(device)
print("Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!")

# 2. Chạy Benchmark chính thức trên tập Test (Chỉ lấy node mới, không tính trùng overlap)
test_f1, test_auc, test_preds, test_labels, test_probs = test_benchmark_new_nodes(
    loader=test_loader, 
    window_size=2000, 
    stride=500
)

AdvancedGAT(
  (lin_in): Linear(in_features=314, out_features=32, bias=True)
  (gat1): GATConv(32, 16, heads=4)
  (gat2): GATConv(64, 32, heads=4)
  (gat3): GATConv(128, 64, heads=4)
  (lin_out): Linear(in_features=64, out_features=11, bias=True)
)
Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...


Epoch: 001/50 | Train Loss: 0.8410 - Macro F1: 0.4488 | Valid Loss: 0.2785 - Macro F1: 0.6692
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 002/50 | Train Loss: 0.3146 - Macro F1: 0.7704 | Valid Loss: 0.1578 - Macro F1: 0.8495
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 003/50 | Train Loss: 0.1559 - Macro F1: 0.8760 | Valid Loss: 0.1173 - Macro F1: 0.9039
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 004/50 | Train Loss: 0.0972 - Macro F1: 0.9212 | Valid Loss: 0.1181 - Macro F1: 0.9016
 -> [EarlyStopping] Bộ đếm tăng: 1 / 8


...

Epoch: 038/50 | Train Loss: 0.0487 - Macro F1: 0.9657 | Valid Loss: 0.0753 - Macro F1: 0.9429
 -> [EarlyStopping] Bộ đếm tăng: 7 / 8


C:\Users\Admin\AppData\Local\Temp\ipykernel_20084\3674275640.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_baseline_gcn.pt'))


Epoch: 039/50 | Train Loss: 0.0545 - Macro F1: 0.9588 | Valid Loss: 0.0918 - Macro F1: 0.9354
 -> [EarlyStopping] Bộ đếm tăng: 8 / 8

🛑 [DỪNG SỚM] Mô hình không cải thiện sau 8 Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch 39!
Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!
Đang chạy Benchmark trên tập Test (Chiến lược: Chỉ đánh giá luồng Mới)...


Test Benchmark: 100%|███████████████████████████████████████████████| 48/48 [00:28<00:00,  1.68it/s]


Tổng số flow thực tế được đánh giá: 760000
🏆 CHÍNH THỨC - TEST BENCHMARK MACRO F1: 0.8669
🏆 CHÍNH THỨC - TEST BENCHMARK AUC-ROC: 0.9959

📊 Classification Report Chi Tiết:
              precision    recall  f1-score   support

           0     0.8881    0.8727    0.8803     19846
           1     0.9965    1.0000    0.9982    484077
           2     0.4263    0.9853    0.5951      2515
           3     0.9987    0.8821    0.9368     36253
           4     0.5429    0.8225    0.6541     18979
           5     0.9796    0.9976    0.9885      2451
           6     0.9339    0.6977    0.7987     11847
           7     1.0000    0.9949    0.9974    107196
           8     0.7944    0.9854    0.8797     16746
           9     1.0000    0.7677    0.8686     41523
          10     0.9681    0.9110    0.9387     18567

    accuracy                         0.9659    760000
   macro avg     0.8662    0.9015    0.8669    760000
weighted avg     0.9750    0.9659    0.9679    760000



In [18]:
# 1. Khởi tạo GAT Model

model = AdvancedGAT(num_node_features=314, num_classes=11)
model = model.to(device)

# 2. Khởi tạo lại Optimizer (Bắt buộc phải tạo lại để nó theo dõi trọng số của GAT)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

# (Hàm criterion NLLLoss với class_weights của bạn vẫn giữ nguyên không đổi)

# 3. Reset Early Stopping (Nên đổi tên file lưu để không ghi đè mất file GCN cũ)
early_stopping = EarlyStopping(patience=8, path='best_advanced_gat.pt') 

print(model)
EPOCHS = 50

# Khởi tạo bộ quản lý Early Stopping với patience = 8
early_stopping = EarlyStopping(patience=8, path='best_baseline_gcn.pt')

print("Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...")
for epoch in range(1, EPOCHS + 1):
    # 1. Chạy huấn luyện và lấy kết quả
    train_loss, train_f1 = train_epoch(epoch, EPOCHS)
    
    # 2. Chạy validation đánh giá
    valid_loss, valid_f1 = valid_epoch(epoch, EPOCHS)
    
    # In kết quả của epoch hiện tại
    print(f"Epoch: {epoch:03d}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} - Macro F1: {train_f1:.4f} | "
          f"Valid Loss: {valid_loss:.4f} - Macro F1: {valid_f1:.4f}")
    
    # 3. Kiểm tra điều kiện dừng sớm
    early_stopping(valid_f1, model)
      
    if early_stopping.early_stop:
        print(f"\n🛑 [DỪNG SỚM] Mô hình không cải thiện sau {early_stopping.patience} Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch {epoch}!")
        break
    
model.load_state_dict(torch.load('best_baseline_gcn.pt'))
model = model.to(device)
print("Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!")

# 2. Chạy Benchmark chính thức trên tập Test (Chỉ lấy node mới, không tính trùng overlap)
test_f1, test_auc, test_preds, test_labels, test_probs = test_benchmark_new_nodes(
    loader=test_loader, 
    window_size=2000, 
    stride=500
)

AdvancedGAT(
  (lin_in): Linear(in_features=314, out_features=32, bias=True)
  (gat1): GATConv(32, 16, heads=4)
  (gat2): GATConv(64, 32, heads=4)
  (gat3): GATConv(128, 64, heads=4)
  (lin_out): Linear(in_features=64, out_features=11, bias=True)
)
Bắt đầu huấn luyện mô hình (Tích hợp Early Stopping)...


Epoch: 001/50 | Train Loss: 0.9188 - Macro F1: 0.4051 | Valid Loss: 0.3346 - Macro F1: 0.6559
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 002/50 | Train Loss: 0.3878 - Macro F1: 0.7102 | Valid Loss: 0.2210 - Macro F1: 0.7728
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 003/50 | Train Loss: 0.2020 - Macro F1: 0.8474 | Valid Loss: 0.2154 - Macro F1: 0.7819
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


Epoch: 004/50 | Train Loss: 0.1458 - Macro F1: 0.8914 | Valid Loss: 0.1157 - Macro F1: 0.8927
 -> [EarlyStopping] 🏆 Phát hiện Macro F1 tốt nhất mới! Đã lưu trọng số mô hình.


...

Epoch: 030/50 | Train Loss: 0.0416 - Macro F1: 0.9652 | Valid Loss: 0.0925 - Macro F1: 0.9345
 -> [EarlyStopping] Bộ đếm tăng: 7 / 8


C:\Users\Admin\AppData\Local\Temp\ipykernel_20084\3304966121.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_baseline_gcn.pt'))


Epoch: 031/50 | Train Loss: 0.0270 - Macro F1: 0.9787 | Valid Loss: 0.1097 - Macro F1: 0.9346
 -> [EarlyStopping] Bộ đếm tăng: 8 / 8

🛑 [DỪNG SỚM] Mô hình không cải thiện sau 8 Epoch liên tiếp. Kích hoạt dừng sớm tại Epoch 31!
Đã khôi phục thành công trọng số mô hình tốt nhất từ file checkpoint!
Đang chạy Benchmark trên tập Test (Chiến lược: Chỉ đánh giá luồng Mới)...


Test Benchmark: 100%|███████████████████████████████████████████████| 48/48 [00:28<00:00,  1.66it/s]


Tổng số flow thực tế được đánh giá: 760000
🏆 CHÍNH THỨC - TEST BENCHMARK MACRO F1: 0.8613
🏆 CHÍNH THỨC - TEST BENCHMARK AUC-ROC: 0.9976

📊 Classification Report Chi Tiết:
              precision    recall  f1-score   support

           0     0.9322    0.8660    0.8979     19846
           1     0.9947    1.0000    0.9973    484077
           2     0.5115    0.9873    0.6739      2515
           3     0.9992    0.8505    0.9189     36253
           4     0.5653    0.7996    0.6623     18979
           5     0.9796    0.9996    0.9895      2451
           6     0.6817    0.7135    0.6972     11847
           7     1.0000    0.9947    0.9974    107196
           8     0.7225    0.9944    0.8369     16746
           9     0.9986    0.7441    0.8527     41523
          10     0.9998    0.9063    0.9508     18567

    accuracy                         0.9627    760000
   macro avg     0.8532    0.8960    0.8613    760000
weighted avg     0.9711    0.9627    0.9643    760000

